# 03 — Dry-run: does cached-branching rows mode actually work?

**Recap from notebook 2**, now backed by the real source code you pulled up (not just my explanation):
- `model.config.text_config.layer_types` literally spells out the pattern: `["linear_attention"]*3 + ["full_attention"]` × 6, confirming `full_attention_interval: 4`.
- The DeltaNet mixer's `forward` source calls `apply_mask_to_padding_states(hidden_states, attention_mask)` — a padding-only use of the mask — then, when a previous cache state exists (`use_precomputed_states`), passes `initial_state=recurrent_state` into `torch_chunk_gated_delta_rule(...)` to continue the recurrence from exactly that starting point. **This confirms our rows-mode design is not a hack we're inventing — "continue the recurrence from a precomputed `initial_state`" is a first-class, intentionally supported code path in this model's own implementation**, used for ordinary incremental decoding and equally valid for our purposes.

**Goal of this notebook:** prove, on the real model, that:
1. Computing `state` once, cloning its cache, and continuing each clone with a different question's tokens gives the **same result** as just running `state + question` from scratch.
2. The two branches don't contaminate each other.

This notebook is self-contained — it reloads the model itself, so you don't need notebook 1 or 2 running first.

In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import copy
import inspect
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "Qwen/Qwen3.5-0.8B"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")
model.eval()
print("loaded.")

loaded.


## Step 0: confirm the *public* forward signature before guessing

Notebook 2 showed the individual mixers use different internal argument names — the DeltaNet mixer takes `cache_params`, the Attention mixer takes `past_key_values`. That's internal routing done for us by the layer stack; we never call either mixer directly. What we actually call is the top-level model's `forward`, and we need to know *its* real argument names before writing code against it — same "verify, don't guess" rule as the rest of this project.

In [ ]:
print(inspect.signature(model.forward))

(input_ids: torch.LongTensor = None, attention_mask: torch.Tensor | None = None, position_ids: torch.LongTensor | None = None, past_key_values: transformers.cache_utils.Cache | None = None, inputs_embeds: torch.FloatTensor | None = None, labels: torch.LongTensor | None = None, pixel_values: torch.Tensor | None = None, pixel_values_videos: torch.FloatTensor | None = None, image_grid_thw: torch.LongTensor | None = None, video_grid_thw: torch.LongTensor | None = None, mm_token_type_ids: torch.IntTensor | None = None, logits_to_keep: int | torch.Tensor = 0, mm_encoder_outputs: dict[str, transformers.modeling_outputs.BaseModelOutputWithPooling] | None = None, **kwargs: Unpack[transformers.utils.generic.TransformersKwargs]) -> tuple | transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5CausalLMOutputWithPast


Confirm `past_key_values` (or whatever it's actually called — adjust the rest of this notebook if it's named differently) and `position_ids` both appear as real parameters, not swallowed into a generic `**kwargs`. If `position_ids` isn't there, that's useful to know too — it may mean the model infers positions automatically from the cache's recorded length, in which case we can likely drop it from our branching code entirely.

## Step 1: build state + two questions, and check the tokenizer boundary

We bypass the chat template here — it's about correct conversational formatting, which isn't what we're testing. We only care about cache-reuse mechanics right now; realistic formatting comes back in notebook 4.

The check below matters: BPE tokenizers can merge text differently depending on what follows it, so `tokenize(state)` is not guaranteed to equal the first `len(state_tokens)` tokens of `tokenize(state + question)`. If this fails, that's a real, useful finding — not a bug in this notebook — and it's a preview of why any2jev uses dedicated delimiter tokens (`<state>`, `<q>`, ...): reserved special tokens can't ambiguously merge with surrounding text the way ordinary subwords can.

In [3]:
tokenizer = processor.tokenizer

state_text = "Customer ticket: payouts have failed three days in a row, and the bank says nothing is wrong on their end."
question_a_text = " Question: which team should handle this — billing, technical, or sales?"
question_b_text = " Question: is this urgent and does it need escalation?"

state_ids = tokenizer(state_text, return_tensors="pt").input_ids.to(model.device)
full_a_ids = tokenizer(state_text + question_a_text, return_tensors="pt").input_ids.to(model.device)
full_b_ids = tokenizer(state_text + question_b_text, return_tensors="pt").input_ids.to(model.device)

L_state = state_ids.shape[1]
print("state length:", L_state, "| full_a length:", full_a_ids.shape[1], "| full_b length:", full_b_ids.shape[1])

prefix_matches_a = torch.equal(full_a_ids[:, :L_state], state_ids)
prefix_matches_b = torch.equal(full_b_ids[:, :L_state], state_ids)
print("full_a's first L_state tokens match state_ids exactly?", prefix_matches_a)
print("full_b's first L_state tokens match state_ids exactly?", prefix_matches_b)

state length: 23 | full_a length: 38 | full_b length: 34
full_a's first L_state tokens match state_ids exactly? True
full_b's first L_state tokens match state_ids exactly? True


**If either printed `False`:** stop and tell me — we'd need a cleaner boundary (e.g. ending `state_text` on a character that can't merge forward) before the rest of this notebook means anything. If both are `True`, continue.

## Step 2: the naive baseline — full forward pass from scratch, no caching tricks

This is the obviously-correct, expensive version: recompute everything from position 0 every time. This is our ground truth to compare against.

In [4]:
with torch.no_grad():
    ref_a = model(input_ids=full_a_ids, output_hidden_states=True)
    ref_b = model(input_ids=full_b_ids, output_hidden_states=True)

ref_a_last_hidden = ref_a.hidden_states[-1][:, -1, :]
ref_b_last_hidden = ref_b.hidden_states[-1][:, -1, :]
print("reference A last-token hidden state shape:", ref_a_last_hidden.shape)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


reference A last-token hidden state shape: torch.Size([1, 1024])


## Step 3: the optimization — compute `state` once, clone the cache, branch per question

Adjust the argument names below (`past_key_values`, `position_ids`) if Step 0 showed the real signature uses something different. We use `copy.deepcopy` for the clone rather than any `.expand()`/view-based shortcut — recall from our discussion that a view would alias the same underlying memory across branches, letting one branch's in-place cache update corrupt the other's.

In [5]:
with torch.no_grad():
    state_out = model(input_ids=state_ids, use_cache=True)
cache_after_state = state_out.past_key_values

cache_a = copy.deepcopy(cache_after_state)
cache_b = copy.deepcopy(cache_after_state)

question_a_only_ids = full_a_ids[:, L_state:]
question_b_only_ids = full_b_ids[:, L_state:]

position_ids_a = torch.arange(L_state, L_state + question_a_only_ids.shape[1], device=model.device).unsqueeze(0)
position_ids_b = torch.arange(L_state, L_state + question_b_only_ids.shape[1], device=model.device).unsqueeze(0)

with torch.no_grad():
    branch_a = model(input_ids=question_a_only_ids, past_key_values=cache_a, position_ids=position_ids_a, use_cache=True, output_hidden_states=True)
    branch_b = model(input_ids=question_b_only_ids, past_key_values=cache_b, position_ids=position_ids_b, use_cache=True, output_hidden_states=True)

branch_a_last_hidden = branch_a.hidden_states[-1][:, -1, :]
branch_b_last_hidden = branch_b.hidden_states[-1][:, -1, :]
print("branch A last-token hidden state shape:", branch_a_last_hidden.shape)

branch A last-token hidden state shape: torch.Size([1, 1024])


**If this cell errors:** paste the traceback. Likely culprits, roughly in order of likelihood: a wrong argument name (fixed by Step 0's real signature), `position_ids` not being the expected shape/dtype, or `copy.deepcopy` not fully detaching something CUDA-side. Any of these is normal to hit on dev-branch code — we'll fix from the real error.

## Step 4: the actual test

In [6]:
print("branch A matches reference A (allclose)?", torch.allclose(branch_a_last_hidden, ref_a_last_hidden, atol=1e-2))
print("branch B matches reference B (allclose)?", torch.allclose(branch_b_last_hidden, ref_b_last_hidden, atol=1e-2))
print("max abs diff A:", (branch_a_last_hidden - ref_a_last_hidden).abs().max().item())
print("max abs diff B:", (branch_b_last_hidden - ref_b_last_hidden).abs().max().item())

branch A matches reference A (allclose)? False
branch B matches reference B (allclose)? False
max abs diff A: 0.375
max abs diff B: 0.25


**How to read this:** we used `allclose` with a loose-ish tolerance (`atol=1e-2`), not exact equality — `bfloat16` arithmetic genuinely produces tiny numerical differences depending on the order operations happen in, even when the *math* being computed is identical. A small max-abs-diff (much smaller than the tolerance) with `allclose=True` means the branch is numerically consistent with the naive baseline — the cached-branching optimization is correct. If `allclose` is `False` or the diff is large, something in the mechanics is wrong (wrong position IDs, a shallow copy instead of a real clone, a missing piece of state) — paste the actual numbers and whatever error occurred, and we'll debug from there rather than assume the design itself was wrong.

If this passes: you've empirically proven, on the real model, that the "smart" rows-mode design from our conversation actually works — not just reasoned about it. Next notebook: designing the actual jev data format (Choice/Score/Noul records, delimiter tokens) now that we know how the backbone will be driven.

In [7]:
num_entries = len(ref_a.hidden_states)
q_len = question_a_only_ids.shape[1]

for i in range(num_entries):
    ref_slice = ref_a.hidden_states[i][:, -q_len:, :]
    branch_slice = branch_a.hidden_states[i]
    diff = (ref_slice - branch_slice).abs().max().item()
    print(f"hidden_states[{i}] max abs diff: {diff:.6f}")

hidden_states[0] max abs diff: 0.000000
hidden_states[1] max abs diff: 0.005859
hidden_states[2] max abs diff: 0.009766
hidden_states[3] max abs diff: 0.011719
hidden_states[4] max abs diff: 0.007812
hidden_states[5] max abs diff: 0.009766
hidden_states[6] max abs diff: 0.007812
hidden_states[7] max abs diff: 0.007812
hidden_states[8] max abs diff: 0.011719
hidden_states[9] max abs diff: 0.011719
hidden_states[10] max abs diff: 0.011719
hidden_states[11] max abs diff: 0.011719
hidden_states[12] max abs diff: 0.011719
hidden_states[13] max abs diff: 0.015625
hidden_states[14] max abs diff: 0.015625
hidden_states[15] max abs diff: 0.015625
hidden_states[16] max abs diff: 0.011719
hidden_states[17] max abs diff: 0.015625
hidden_states[18] max abs diff: 0.023438
hidden_states[19] max abs diff: 0.031250
hidden_states[20] max abs diff: 0.046875
hidden_states[21] max abs diff: 0.062500
hidden_states[22] max abs diff: 0.062500
hidden_states[23] max abs diff: 0.062500
hidden_states[24] max abs 

In [8]:
ref_final = ref_a.hidden_states[24][:, -q_len:, :]
print("typical magnitude (mean abs):", ref_final.abs().mean().item())
print("max magnitude:", ref_final.abs().max().item())
rel_diff = (ref_final - branch_a.hidden_states[24]).abs() / (ref_final.abs() + 1e-6)
print("relative diff — max:", rel_diff.max().item(), "mean:", rel_diff.mean().item())

typical magnitude (mean abs): 2.90625
max magnitude: 36.0
relative diff — max: 81920.0 mean: 11.3125


In [9]:
with torch.no_grad():
    ref_a_repeat = model(input_ids=full_a_ids, output_hidden_states=True)
noise_floor = (ref_a.hidden_states[-1][:, -1, :] - ref_a_repeat.hidden_states[-1][:, -1, :]).abs().max().item()
print("noise floor from repeating the identical call:", noise_floor)

noise floor from repeating the identical call: 0.0


In [10]:
model_inputs_a = model.prepare_inputs_for_generation(
    input_ids=full_a_ids,
    past_key_values=cache_a,
    use_cache=True,
)
print(model_inputs_a.keys())


dict_keys(['input_ids', 'past_key_values', 'use_cache'])


In [11]:
print(model_inputs_a["input_ids"].shape)
print(model_inputs_a["input_ids"])


torch.Size([1, 38])
tensor([[12425, 11390,    25, 86449,   599,  4490,  2250,  2756,   303,   264,
          2713,    11,   321,   279,  5883,  2640,  4161,   369,  4808,   383,
           836,   809,    13, 15380,    25,   864,  2015,  1220,  3579,   411,
          1892, 32419,    11, 10597,    11,   466,  6416,    30]],
       device='cuda:0')


In [12]:
cache_a2 = copy.deepcopy(cache_after_state)
cache_b2 = copy.deepcopy(cache_after_state)

with torch.no_grad():
    branch_a2 = model(input_ids=question_a_only_ids, past_key_values=cache_a2, use_cache=True, output_hidden_states=True)
    branch_b2 = model(input_ids=question_b_only_ids, past_key_values=cache_b2, use_cache=True, output_hidden_states=True)

branch_a2_last_hidden = branch_a2.hidden_states[-1][:, -1, :]
branch_b2_last_hidden = branch_b2.hidden_states[-1][:, -1, :]

print("branch A (no manual position_ids) matches reference A?", torch.allclose(branch_a2_last_hidden, ref_a_last_hidden, atol=1e-2))
print("max abs diff A:", (branch_a2_last_hidden - ref_a_last_hidden).abs().max().item())
print("branch B (no manual position_ids) matches reference B?", torch.allclose(branch_b2_last_hidden, ref_b_last_hidden, atol=1e-2))
print("max abs diff B:", (branch_b2_last_hidden - ref_b_last_hidden).abs().max().item())


branch A (no manual position_ids) matches reference A? False
max abs diff A: 0.375
branch B (no manual position_ids) matches reference B? False
max abs diff B: 0.25


In [13]:
print(cache_a.has_previous_state(0, state_idx=0))
print(cache_a.layers[0].conv_states[0].abs().sum().item())
print(cache_after_state.layers[0].conv_states[0].abs().sum().item())


True
18176.0
17536.0


In [14]:
fresh_check = copy.deepcopy(cache_after_state)
print("conv_states match:", torch.equal(fresh_check.layers[0].conv_states[0], cache_after_state.layers[0].conv_states[0]))
print("recurrent_states match:", torch.equal(fresh_check.layers[0].recurrent_states[0], cache_after_state.layers[0].recurrent_states[0]))


conv_states match: True
recurrent_states match: True


In [15]:
print("=== update_conv_state source ===")
print(inspect.getsource(type(cache_after_state).update_conv_state))
print()
print("=== update_recurrent_state source ===")
print(inspect.getsource(type(cache_after_state).update_recurrent_state))


=== update_conv_state source ===
    def update_conv_state(
        self, conv_states: torch.Tensor, layer_idx: int, state_idx: int = 0, **kwargs
    ) -> torch.Tensor:
        """
        Updates the cache with the new `conv_states` for the layer `layer_idx`.

        Parameters:
            conv_states (`torch.Tensor`):
                The new conv states to cache.
            layer_idx (`int`):
                The index of the layer to cache the states for.

        Return:
            `torch.Tensor`: The updated conv states.
        """
        # NOTE: if we slightly break `update` arg order, we could combine this with it, and allow offloading support
        # out of the box
        if not isinstance(self.layers[layer_idx], LinearAttentionCacheLayerMixin):
            raise ValueError("Cannot call `update_conv_state` on a non-LinearAttention layer!")
        conv_states = self.layers[layer_idx].update_conv_state(conv_states, state_idx, **kwargs)
        return conv_states


=== up

In [16]:
print("=== layer-level update_conv_state source ===")
print(inspect.getsource(type(cache_after_state.layers[0]).update_conv_state))
print()
print("=== layer-level update_recurrent_state source ===")
print(inspect.getsource(type(cache_after_state.layers[0]).update_recurrent_state))


=== layer-level update_conv_state source ===
    def update_conv_state(
        self, conv_states: torch.Tensor, state_idx: int = 0, conv_kernel_size: int | None = None, **kwargs
    ) -> torch.Tensor:
        """
        Update the linear attention cache in-place, and return the necessary conv states.

        Args:
            conv_states (`torch.Tensor`): The new conv states to cache.

        Returns:
            `torch.Tensor`: The updated conv states.
        """
        # Lazy initialization
        if not self.is_conv_states_initialized[state_idx]:
            self.lazy_initialization(conv_states=conv_states, state_idx=state_idx, conv_kernel_size=conv_kernel_size)

        # This is prefill, simply pad the conv_states if necessary
        if not self.has_previous_state[state_idx]:
            full_conv_states = conv_states
            self.has_previous_state[state_idx] = True
            # In this case, need to pad it to fit the conv_kernel_size
            if not self.record_p

In [17]:
# Find the actual classes from the layer list in notebook 1.
layers = model.model.language_model.layers
delta_layer = layers[0]
attn_layer = layers[3]

delta_mixer = [m for n, m in delta_layer.named_children()][0]
attn_mixer = [m for n, m in attn_layer.named_children()][0]

In [18]:
mod = inspect.getmodule(type(delta_mixer))
print(inspect.getsource(mod.causal_conv1d_fn))


@use_kernel_func_from_hub_with_fallback("causal_conv1d_fn", "causal_conv1d")
def causal_conv1d_fn(
    hidden_states: torch.Tensor,
    weight: nn.Parameter,
    bias: nn.Parameter | None = None,
    activation: str | None = None,
    **kwargs,
):
    _, hidden_size, seq_len = hidden_states.shape
    padding = weight.shape[-1] - 1

    out = F.conv1d(
        hidden_states.to(weight.dtype),
        weight=weight.unsqueeze(1),
        bias=bias,
        padding=padding,
        groups=hidden_size,
    )[:, :, :seq_len]
    if activation is not None:
        out = ACT2FN[activation](out)
    return out.to(hidden_states.dtype)



In [19]:
def run_branch(question_ids, base_cache):
    cache = copy.deepcopy(base_cache)
    out = None
    with torch.no_grad():
        for t in range(question_ids.shape[1]):
            tok = question_ids[:, t:t+1]
            out = model(input_ids=tok, past_key_values=cache, use_cache=True, output_hidden_states=True)
    return out.hidden_states[-1][:, -1, :]   # last-token hidden after the whole question

branch_a_last_hidden = run_branch(question_a_only_ids, cache_after_state)
branch_b_last_hidden = run_branch(question_b_only_ids, cache_after_state)

print("A matches ref?", torch.allclose(branch_a_last_hidden, ref_a_last_hidden, atol=1e-2),
      "| max diff:", (branch_a_last_hidden - ref_a_last_hidden).abs().max().item())
print("B matches ref?", torch.allclose(branch_b_last_hidden, ref_b_last_hidden, atol=1e-2),
      "| max diff:", (branch_b_last_hidden - ref_b_last_hidden).abs().max().item())


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


A matches ref? False | max diff: 0.25
B matches ref? False | max diff: 0.21875


In [20]:
with torch.no_grad():
    batch_out = model(input_ids=torch.cat([full_a_ids, full_b_ids_padded := torch.nn.functional.pad(
        full_b_ids, (0, full_a_ids.shape[1]-full_b_ids.shape[1]), value=tokenizer.pad_token_id or 0)], dim=0),
        output_hidden_states=True)


In [21]:
with torch.no_grad():
    rows_a = model(input_ids=full_a_ids, output_hidden_states=True).hidden_states[-1][:, -1, :]
    rows_b = model(input_ids=full_b_ids, output_hidden_states=True).hidden_states[-1][:, -1, :]

# these ARE the reference by construction — the point is this is our rows-mode:
# one independent full pass per question, state recomputed each time.
print("rows_a == ref_a?", torch.equal(rows_a, ref_a_last_hidden))
print("rows_b == ref_b?", torch.equal(rows_b, ref_b_last_hidden))


rows_a == ref_a? True
rows_b == ref_b? True
